In [ ]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_benchmark.py"
github_token = None

# Drive remains the home for images and benchmark outputs.
drive.mount("/content/drive")


def run_git(args, check=True):
    """Run git and print useful stderr without leaking the optional token."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(github_token, "***") if github_token else result.stdout
    stderr = result.stderr.replace(github_token, "***") if github_token else result.stderr
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        hint = ""
        if "403" in stderr or "Write access to repository not granted" in stderr:
            hint = (
                " The GITHUB_TOKEN secret exists, but GitHub rejected it. "
                "Create a token that has read access to this private repository "
                "and permission to read repository contents."
            )
        raise RuntimeError(f"Git command failed with exit code {result.returncode}: {args[:2]}.{hint}")
    return result


# Colab does not automatically check out the full GitHub repo when opening a notebook.
# For this private repo, add a Colab secret named GITHUB_TOKEN only if unauthenticated clone fails.
if (REPO_DIR / ".git").exists():
    pull_result = run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    if pull_result.returncode != 0:
        print("WARNING: git pull failed; continuing with the existing Colab checkout.")
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Restart the Colab runtime or remove that folder, then rerun this setup cell."
        )

    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs a Colab secret named GITHUB_TOKEN "
            "so the Colab runtime can clone src/radle_benchmark.py. "
            "Add the secret, restart or rerun this setup cell, then continue."
        )

    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")
    run_git(["git", "clone", clone_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Benchmark module not found after setup: {MODULE_PATH}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Ensure the Anthropic SDK is available.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic", "google-genai"], check=True)


In [ ]:
# ==========================================
# 2. API CLIENT + BENCHMARK IMPORTS
# ==========================================
import importlib
import inspect

import anthropic
import radle_benchmark

radle_benchmark = importlib.reload(radle_benchmark)
create_scorer_view = radle_benchmark.create_scorer_view
run_benchmark = radle_benchmark.run_benchmark
audit_benchmark_output = radle_benchmark.audit_benchmark_output
run_targeted_repair = radle_benchmark.run_targeted_repair

for _required_param in ("anthropic_client", "gemini_client", "backup_dir"):
    if _required_param not in inspect.signature(run_benchmark).parameters:
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_param}. "
            "Rerun the setup/import cells after git pull, or restart the runtime."
        )

for _required_function in (
    "audit_benchmark_output",
    "run_targeted_repair",
    "build_run_paths",
    "promote_final_results",
    "export_public_release_tables",
):
    if not hasattr(radle_benchmark, _required_function):
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_function}. "
            "Rerun the setup/import cells after git pull, or restart the runtime."
        )


# Fable 5 has always-on adaptive thinking. The repo adapter currently couples
# output_config to an explicit thinking block for Opus 4.8, so this copied
# notebook patches only Fable requests to omit temperature/thinking and keep effort=high.
_original_build_api_params = radle_benchmark.build_api_params


def _build_api_params_with_fable(model, content_array, max_output_tokens, universal_temperature):
    if model.get("provider") == "anthropic" and model.get("id") == "claude-fable-5":
        extra = model.get("extra") or {}
        params = {
            "model": model["id"],
            "max_tokens": max_output_tokens,
            "messages": [
                {
                    "role": "user",
                    "content": radle_benchmark._convert_content_for_anthropic(content_array),
                }
            ],
        }
        if "output_config" in extra:
            params["output_config"] = extra["output_config"]
        return params

    return _original_build_api_params(
        model,
        content_array,
        max_output_tokens,
        universal_temperature,
    )


radle_benchmark.build_api_params = _build_api_params_with_fable

_original_extract_result = radle_benchmark.extract_result


def _extract_result_with_anthropic_status(response, latency, api_params, grok_fallback_used, model):
    result = _original_extract_result(response, latency, api_params, grok_fallback_used, model)
    if model.get("provider") == "anthropic":
        name = model["name"]
        result[f"Returned_Model_{name}"] = getattr(response, "model", "")
        result[f"Anthropic_Stop_Reason_{name}"] = getattr(response, "stop_reason", "") or ""
        result[f"Anthropic_Stop_Details_{name}"] = radle_benchmark.json.dumps(
            radle_benchmark.make_json_safe(getattr(response, "stop_details", None)),
            ensure_ascii=False,
        )
    return result


radle_benchmark.extract_result = _extract_result_with_anthropic_status

print(f"Loaded benchmark module: {radle_benchmark.__file__}")

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    print("ERROR: Could not find ANTHROPIC_API_KEY in Colab Secrets.")

anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# This Fable copy is native Anthropic only. Keep the unused generic clients as
# None so the benchmark signature is satisfied without non-Anthropic routing.
openai_client = None
gemini_client = None
openrouter_client = None


In [ ]:
# ==========================================
# 3. DRIVE PATHS + FABLE 5 RUN CONFIG
# ==========================================
from pathlib import Path

# Five-case Fable smoke run. Use TEST_LIMIT=None only after this copy is intentionally promoted.
TEST_LIMIT = 5
RUN_LABEL = "claude_fable_5_test_5_cases"

FABLE_MODEL = {
    "name": "claude_fable_5",
    "id": "claude-fable-5",
    "provider": "anthropic",
    "extra": {"output_config": {"effort": "high"}},
}
ACTIVE_MODELS = [FABLE_MODEL]
if any(model.get("provider") != "anthropic" for model in ACTIVE_MODELS):
    raise RuntimeError("This Fable notebook is native Anthropic only; OpenRouter models are not allowed here.")

dataset_root = Path("/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset")
run_paths = radle_benchmark.build_run_paths(dataset_root, run_label=RUN_LABEL)

master_images_folder = run_paths["master_images_folder"]
final_output_csv = run_paths["raw_results_csv"]
raw_backup_dir = run_paths["raw_backup_dir"]
scorer_csv = run_paths["scorer_view_csv"]
repair_output_csv = run_paths["repair_results_csv"]
repair_call_log_csv = run_paths["repair_call_log_csv"]
repair_plan_csv = run_paths["repair_plan_csv"]
repair_backup_dir = run_paths["repair_backup_dir"]
final_results_csv = run_paths["final_results_csv"]
final_manifest_json = run_paths["final_manifest_json"]
public_release_dir = run_paths["public_release_dir"]

print("Run label:", RUN_LABEL)
print("Run folder:", run_paths["run_root"])
print("Raw results CSV:", final_output_csv)
print("Running models:", [m["name"] for m in ACTIVE_MODELS])
print("Model IDs:", {m["name"]: m["id"] for m in ACTIVE_MODELS})


In [ ]:
# ==========================================
# 4. RUN BENCHMARK
# ==========================================
df_final = run_benchmark(
    client=None,  # Native Anthropic route; no OpenRouter client is used.
    openai_client=None,
    anthropic_client=anthropic_client,
    gemini_client=None,
    image_folder=master_images_folder,
    output_csv=final_output_csv,
    test_limit=TEST_LIMIT,
    models=ACTIVE_MODELS,
    backup_dir=raw_backup_dir,
)

print("\nFINAL DATAFRAME PREVIEW:")
from IPython.display import display

display(df_final.head())


In [ ]:
# ==========================================
# 4.5. VERIFY RETURNED MODEL ID
# ==========================================
expected_model_id = FABLE_MODEL["id"]
returned_model_col = f"Returned_Model_{FABLE_MODEL['name']}"
legacy_returned_model_col = f"OpenRouter_Response_Model_{FABLE_MODEL['name']}"

model_id_columns = [
    col for col in (returned_model_col, legacy_returned_model_col) if col in df_final.columns
]
if not model_id_columns:
    raise RuntimeError(
        "No returned-model column was written. Cannot prove whether Fable or a fallback model handled the run."
    )

for col in model_id_columns:
    returned_models = sorted(
        {str(value).strip() for value in df_final[col].dropna().unique() if str(value).strip()}
    )
    print(f"{col}:", returned_models)
    if returned_models != [expected_model_id]:
        raise RuntimeError(
            f"Expected returned model {expected_model_id}, but {col} contained {returned_models}. "
            "Treat this run as not clean Fable evidence until the model routing is resolved."
        )

stop_reason_col = f"Anthropic_Stop_Reason_{FABLE_MODEL['name']}"
stop_details_col = f"Anthropic_Stop_Details_{FABLE_MODEL['name']}"
if stop_reason_col in df_final.columns:
    print(f"{stop_reason_col}:", sorted(df_final[stop_reason_col].dropna().astype(str).unique()))
if stop_details_col in df_final.columns:
    print(f"{stop_details_col}:", sorted(df_final[stop_details_col].dropna().astype(str).unique()))


In [ ]:
# ==========================================
# 5. SCORER'S VIEW & PIVOT
# ==========================================
df_scorer, display_df, scorer_csv = create_scorer_view(final_output_csv, scorer_csv=scorer_csv)

print(f"Scorer version saved to: {scorer_csv}")
print("\nTRANSPOSED CONSENSUS VIEW:")
display(display_df)


In [ ]:
# ==========================================
# 6. READ-ONLY OUTPUT AUDIT
# ==========================================
expected_case_ids = range(1, TEST_LIMIT + 1) if TEST_LIMIT is not None else range(1, 201)

audit_results = audit_benchmark_output(
    raw_csv=final_output_csv,
    models=ACTIVE_MODELS,
    expected_case_ids=expected_case_ids,
)

print("DATASET INTEGRITY:")
display(audit_results["dataset_integrity"])

print("\nBUCKET SUMMARY:")
display(audit_results["bucket_summary"])

print("\nSTATUS SUMMARY:")
display(audit_results["status_summary"])

print("\nNO-API CLEANUP TARGETS:")
display(audit_results["no_paid_cleanup"].head(100))

print("\nREPAIR TARGETS:")
display(audit_results["repair_targets"].head(100))

print("\nANALYSIS FLAGS")
display(audit_results["analysis_flags"].head(100))

print("\nPROVIDER CONTENT BLOCKS:")
display(audit_results["provider_content_blocks"].head(100))


In [ ]:
# ==========================================
# 7. TARGETED REPAIR PLAN / RUN
# ==========================================
# Keep REPAIR_CONFIRMATION="NO" to preview the repair plan without API calls or file writes.
# Change to "YES_REPAIR_10" for a capped repair test or "YES_REPAIR_ALL" for all eligible targets.
REPAIR_CONFIRMATION = "NO"

repair_results = run_targeted_repair(
    client=None,  # Native Anthropic route; no OpenRouter client is used.
    openai_client=None,
    anthropic_client=anthropic_client,
    gemini_client=None,
    image_folder=master_images_folder,
    input_csv=final_output_csv,
    output_csv=repair_output_csv,
    repair_call_log_csv=repair_call_log_csv,
    repair_plan_csv=repair_plan_csv,
    confirmation=REPAIR_CONFIRMATION,
    models=ACTIVE_MODELS,
    backup_dir=repair_backup_dir,
)

print("\nNO-API CLEANUP PLAN PREVIEW:")
display(repair_results["no_paid_cleanup_plan"].head(100))

print("\nREPAIR PLAN PREVIEW:")
display(repair_results["repair_plan"].head(100))
print("API calls this run:", repair_results["api_calls_this_run"])
print("No-API cleanups applied:", repair_results["no_paid_cleanups_applied"])
print("Repair input CSV:", final_output_csv)
print("Repair output CSV:", repair_results["output_csv"])
print("Repair call log CSV:", repair_results["repair_call_log_csv"])


In [ ]:
# ==========================================
# 8. PROMOTE PRIVATE FINAL FILE
# ==========================================
repair_confirmed = globals().get("REPAIR_CONFIRMATION", "NO") != "NO"
repair_output_ready = repair_confirmed and Path(repair_output_csv).exists()
private_final_source_csv = repair_output_csv if repair_output_ready else final_output_csv
private_final_source_label = "repaired" if repair_output_ready else "raw"

final_manifest = radle_benchmark.promote_final_results(
    source_csv=private_final_source_csv,
    final_csv=final_results_csv,
    manifest_json=final_manifest_json,
    run_id=run_paths["run_id"],
    source_label=private_final_source_label,
    metadata={
        "run_label": RUN_LABEL,
        "test_limit": TEST_LIMIT if TEST_LIMIT is not None else "full",
        "models": [m["name"] for m in ACTIVE_MODELS],
        "model_ids": {m["name"]: m["id"] for m in ACTIVE_MODELS},
    },
)

print("Private final source:", private_final_source_csv)
print("Private final CSV:", final_results_csv)
print("Private final manifest:", final_manifest_json)
print("Private final SHA256:", final_manifest["sha256"])

# Rebuild the scorer view from the repaired/promoted final so it reflects repairs.
if Path(final_results_csv).exists():
    radle_benchmark.create_scorer_view(final_results_csv, scorer_csv=scorer_csv)
    print("Scorer view rebuilt from repaired final:", scorer_csv)


In [ ]:
# ==========================================
# 9. EXPORT ANSWER-FREE PUBLIC RELEASE TABLES
# ==========================================
public_results_source_csv = final_results_csv if Path(final_results_csv).exists() else final_output_csv
public_call_log_csv = repair_call_log_csv if Path(repair_call_log_csv).exists() else None

public_release_files = radle_benchmark.export_public_release_tables(
    results_csv=public_results_source_csv,
    output_dir=public_release_dir,
    models=ACTIVE_MODELS,
    call_log_csv=public_call_log_csv,
    run_id=run_paths["run_id"],
)

print("Public release source CSV:", public_results_source_csv)
print("Public case-model CSV:", public_release_files["case_model_csv"])
print("Public model summary CSV:", public_release_files["summary_csv"])
print("Public sanitized call log CSV:", public_release_files["sanitized_call_log_csv"])
print("Public manifest:", public_release_files["manifest_json"])
